# Systematic Feature Selection: Weekly & Lagged Predictors of Cattle Mortality

## Objective

Systematically identify the **optimal subset of weekly heat stress predictors** for cattle mortality in USDA Regions 4 & 6 through rigorous comparison of 15 feature selection approaches.

## Scientific Framework

**Challenge**: We have ~60 potential predictors including:
- Base weekly metrics (6)
- Lagged features (24): 1-4 week lags
- Rolling averages (12): 2-week and 4-week windows
- Interaction terms (2)
- Temporal controls (6+)

**Problem**: High multicollinearity, risk of overfitting, interpretability challenges

**Solution**: Test multiple feature selection methods and compare using:
- **AIC/BIC**: Information criteria balancing fit and complexity
- **Cross-validation**: Honest estimate of generalization
- **Adjusted R²**: Variance explained with complexity penalty
- **Parsimony**: Fewer features for similar performance

## Hypotheses

**H1**: LASSO-based methods will outperform stepwise selection (better handling of correlated features)

**H2**: Lagged features (1-2 weeks) will improve predictions over current-week-only models

**H3**: Final model will contain 8-15 features (parsimonious but informative)

**H4**: VPD, extreme heat (>35°C), and poor nighttime recovery will be consistently selected

---

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Machine learning
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LassoCV, ElasticNetCV
from sklearn.feature_selection import RFE, RFECV, mutual_info_regression
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Statistical modeling (optional - for advanced diagnostics)
try:
    import statsmodels.api as sm
    import statsmodels.formula.api as smf
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    from statsmodels.stats.diagnostic import het_breuschpagan, acorr_ljungbox
    from statsmodels.stats.stattools import durbin_watson
    STATSMODELS_AVAILABLE = True
    print("✓ statsmodels loaded successfully")
except ImportError as e:
    STATSMODELS_AVAILABLE = False
    print("⚠ Warning: statsmodels not available")
    print("  Some advanced diagnostics (VIF, Breusch-Pagan test) will be skipped")
    print("  To enable: pip install --upgrade statsmodels scipy")
    print(f"  Error: {e}")

import pickle

# Configuration
import sys
sys.path.append('../../')
from config import (
    PROCESSED_WEEKLY_DIR, MASK_FILE, CATTLE_DATA_FILE,
    CATTLE_REGION_IDS, SEASONS, FOCUS_STATES, STATE_ABBRS
)

# Plotting
sns.set_style('whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 11

# Reproducibility
np.random.seed(42)

# Output directory
OUTPUT_DIR = Path('../../figures/feature_selection_weekly')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("\\nCore libraries loaded successfully")
print(f"Statsmodels available: {STATSMODELS_AVAILABLE}")

## 1. Data Loading and Feature Engineering

In [ ]:
# Load climate data
print("Loading weekly climate data...")
ds_night = xr.open_dataset(PROCESSED_WEEKLY_DIR / 'weekly_nighttime_recovery.nc')
ds_day = xr.open_dataset(PROCESSED_WEEKLY_DIR / 'weekly_daytime_heat.nc')
ds_vpd = xr.open_dataset(PROCESSED_WEEKLY_DIR / 'weekly_vpd.nc')

week_dates = ds_night['week'].values

# Load state mask
ds_mask = xr.open_dataset(MASK_FILE)
state_mask = ds_mask.state_mask.load()
ds_mask.close()

# Helper function
def compute_regional_mean(data, state_ids, state_mask):
    combined_mask = xr.zeros_like(state_mask, dtype=bool)
    for state_id in state_ids:
        combined_mask = combined_mask | (state_mask == state_id)
    masked_data = data.where(combined_mask)
    return masked_data.mean(dim=['lat', 'lon']).astype(np.float64)

region_4_ids = CATTLE_REGION_IDS['region_4']
region_6_ids = CATTLE_REGION_IDS['region_6']

# Compute regional metrics for both regions
print("Computing regional climate metrics...")
climate_dfs = []

for region_num, state_ids in [(4, region_4_ids), (6, region_6_ids)]:
    region_df = pd.DataFrame({
        'week_ending': pd.to_datetime(week_dates),
        'region': region_num,
        'mean_daytime_hours_above_30': compute_regional_mean(ds_day['hours_above_30'], state_ids, state_mask).values,
        'mean_daytime_hours_above_35': compute_regional_mean(ds_day['hours_above_35'], state_ids, state_mask).values,
        'mean_nighttime_hours_above_21': compute_regional_mean(ds_night['hours_above_21'], state_ids, state_mask).values,
        'mean_nighttime_hours_above_24': compute_regional_mean(ds_night['hours_above_24'], state_ids, state_mask).values,
        'mean_vpd_mean': compute_regional_mean(ds_vpd['vpd_mean'], state_ids, state_mask).values,
        'mean_vpd_max': compute_regional_mean(ds_vpd['vpd_max'], state_ids, state_mask).values,
    })
    climate_dfs.append(region_df)

climate_data = pd.concat(climate_dfs, ignore_index=True)

# Load cattle data
print("Loading cattle mortality data...")
cattle_data = pd.read_csv(CATTLE_DATA_FILE, parse_dates=['date'])
cattle_data = cattle_data.rename(columns={'date': 'week_ending'})

# Reshape for regions
cattle_r4 = cattle_data[['week_ending']].copy()
cattle_r4['region'] = 4
cattle_r4['total_beef_dairy'] = cattle_data['region_4_beef_dairy']

cattle_r6 = cattle_data[['week_ending']].copy()
cattle_r6['region'] = 6
cattle_r6['total_beef_dairy'] = cattle_data['region_6_beef_dairy']

cattle_regional = pd.concat([cattle_r4, cattle_r6], ignore_index=True)

# Merge
cattle_heat = pd.merge(cattle_regional, climate_data, on=['week_ending', 'region'], how='inner')
cattle_heat = cattle_heat.dropna()

print(f"\nMerged data: {len(cattle_heat)} region-weeks")
print(f"Date range: {cattle_heat['week_ending'].min()} to {cattle_heat['week_ending'].max()}")

In [ ]:
# Add temporal features
cattle_heat['year'] = cattle_heat['week_ending'].dt.year
cattle_heat['month'] = cattle_heat['week_ending'].dt.month
cattle_heat['week_of_year'] = cattle_heat['week_ending'].dt.isocalendar().week

def get_season(month):
    if month in SEASONS['Winter']:
        return 'Winter'
    elif month in SEASONS['Spring']:
        return 'Spring'
    elif month in SEASONS['Summer']:
        return 'Summer'
    else:
        return 'Fall'

cattle_heat['season'] = cattle_heat['month'].apply(get_season)

# Sort by region and date
cattle_heat = cattle_heat.sort_values(['region', 'week_ending']).reset_index(drop=True)

# Create engineered features
print("\nCreating engineered features...")

heat_metrics = [
    'mean_daytime_hours_above_30',
    'mean_daytime_hours_above_35',
    'mean_nighttime_hours_above_21',
    'mean_nighttime_hours_above_24',
    'mean_vpd_mean',
    'mean_vpd_max'
]

# Lagged features (1-4 weeks)
for lag in range(1, 5):
    for metric in heat_metrics:
        cattle_heat[f'{metric}_lag{lag}'] = cattle_heat.groupby('region')[metric].shift(lag)

print(f"  Created {4 * len(heat_metrics)} lagged features")

# Rolling averages (2, 4 week windows)
for window in [2, 4]:
    for metric in heat_metrics:
        cattle_heat[f'{metric}_roll{window}'] = cattle_heat.groupby('region')[metric].transform(
            lambda x: x.rolling(window=window, min_periods=1).mean()
        )

print(f"  Created {2 * len(heat_metrics)} rolling average features")

# Interaction terms
cattle_heat['heat_vpd_interaction'] = (
    cattle_heat['mean_daytime_hours_above_30'] * cattle_heat['mean_vpd_mean']
)
cattle_heat['extreme_heat_poor_recovery'] = (
    cattle_heat['mean_daytime_hours_above_35'] * cattle_heat['mean_nighttime_hours_above_21']
)

print("  Created 2 interaction features")

# One-hot encode season
cattle_heat = pd.get_dummies(cattle_heat, columns=['season'], prefix='season', drop_first=False)

print(f"\nTotal columns: {len(cattle_heat.columns)}")

## 2. Prepare Data for Modeling

In [ ]:
# Define feature sets
target_var = 'total_beef_dairy'

# Base features
base_features = heat_metrics.copy()

# Lagged features
lagged_features = [col for col in cattle_heat.columns if 'lag' in col]

# Rolling features
rolling_features = [col for col in cattle_heat.columns if 'roll' in col]

# Interaction features
interaction_features = ['heat_vpd_interaction', 'extreme_heat_poor_recovery']

# Temporal features
temporal_features = ['month', 'week_of_year'] + [col for col in cattle_heat.columns if col.startswith('season_')]

# Regional feature
regional_features = ['region']

# All features
all_features = (base_features + lagged_features + rolling_features + 
                interaction_features + temporal_features + regional_features)

print(f"Feature breakdown:")
print(f"  Base metrics: {len(base_features)}")
print(f"  Lagged: {len(lagged_features)}")
print(f"  Rolling: {len(rolling_features)}")
print(f"  Interactions: {len(interaction_features)}")
print(f"  Temporal: {len(temporal_features)}")
print(f"  Regional: {len(regional_features)}")
print(f"  TOTAL: {len(all_features)}")

In [ ]:
# Prepare modeling data
model_data = cattle_heat[all_features + [target_var, 'year']].dropna().copy()

print(f"Data after removing NaN: {len(model_data)} samples")

# Time-based split
train_mask = model_data['year'] <= 2015
test_mask = model_data['year'] > 2015

X_train = model_data.loc[train_mask, all_features]
y_train = model_data.loc[train_mask, target_var]
X_test = model_data.loc[test_mask, all_features]
y_test = model_data.loc[test_mask, target_var]

print(f"\nTrain set: {len(X_train)} samples (years {model_data.loc[train_mask, 'year'].min()}-{model_data.loc[train_mask, 'year'].max()})")
print(f"Test set: {len(X_test)} samples (years {model_data.loc[test_mask, 'year'].min()}-{model_data.loc[test_mask, 'year'].max()})")

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nFeatures standardized")

## 3. Baseline Models

In [ ]:
# Helper function to calculate metrics
def calculate_metrics(y_true, y_pred, n_features, n_samples):
    """Calculate comprehensive model metrics"""
    r2 = r2_score(y_true, y_pred)
    adj_r2 = 1 - (1 - r2) * (n_samples - 1) / (n_samples - n_features - 1)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    
    # Calculate AIC and BIC
    n = n_samples
    k = n_features + 1  # +1 for intercept
    rss = np.sum((y_true - y_pred) ** 2)
    
    aic = n * np.log(rss / n) + 2 * k
    bic = n * np.log(rss / n) + k * np.log(n)
    
    return {
        'R2': r2,
        'Adj_R2': adj_r2,
        'RMSE': rmse,
        'MAE': mae,
        'AIC': aic,
        'BIC': bic,
        'N_Features': n_features
    }

# Initialize results storage
all_results = []

# Model 0: Null model (mean only)
print("\n" + "="*80)
print("BASELINE MODELS")
print("="*80)

y_train_mean = y_train.mean()
y_test_pred_null = np.full(len(y_test), y_train_mean)

null_metrics = calculate_metrics(y_test, y_test_pred_null, 0, len(y_test))
null_metrics['Model'] = 'Null (Mean Only)'
null_metrics['Method'] = 'Baseline'
null_metrics['Selected_Features'] = '[]'
all_results.append(null_metrics)

print(f"\nModel 0: Null (Mean Only)")
print(f"  Test RMSE: {null_metrics['RMSE']:.2f}")
print(f"  Test R²: {null_metrics['R2']:.4f}")

# Model 1: Full model (all features)
lr_full = LinearRegression()
lr_full.fit(X_train_scaled, y_train)
y_test_pred_full = lr_full.predict(X_test_scaled)

full_metrics = calculate_metrics(y_test, y_test_pred_full, len(all_features), len(y_test))
full_metrics['Model'] = 'Full Model'
full_metrics['Method'] = 'Baseline'
full_metrics['Selected_Features'] = str(all_features)
all_results.append(full_metrics)

print(f"\nModel 1: Full Model (All {len(all_features)} features)")
print(f"  Test RMSE: {full_metrics['RMSE']:.2f}")
print(f"  Test R²: {full_metrics['R2']:.4f}")
print(f"  Test Adj R²: {full_metrics['Adj_R2']:.4f}")
print(f"  AIC: {full_metrics['AIC']:.0f}")
print(f"  BIC: {full_metrics['BIC']:.0f}")